# 🟡 Solution: Classifier-Free Guidance

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def classifier_free_guidance(eps_uncond, eps_cond, guidance_scale, rescale=0.0):
    # Linear extrapolation away from the unconditional prediction
    eps_cfg = eps_uncond + guidance_scale * (eps_cond - eps_uncond)

    if rescale > 0.0:
        # Per-sample statistics: every dim except the batch dim
        dims = tuple(range(1, eps_cfg.dim()))
        std_cond = eps_cond.std(dim=dims, keepdim=True)
        std_cfg = eps_cfg.std(dim=dims, keepdim=True)

        # Renormalise back to the conditional branch's scale, then blend
        eps_rescaled = eps_cfg * (std_cond / std_cfg)
        eps_cfg = rescale * eps_rescaled + (1.0 - rescale) * eps_cfg

    return eps_cfg

In [ ]:
# Verify
eps_u = torch.randn(4, 3, 8, 8)
eps_c = torch.randn(4, 3, 8, 8)

out = classifier_free_guidance(eps_u, eps_c, 7.5)
print("w=7.5 matches formula :", torch.allclose(out, eps_u + 7.5 * (eps_c - eps_u), atol=1e-6))
print("w=1.0 is conditional  :", torch.allclose(classifier_free_guidance(eps_u, eps_c, 1.0), eps_c, atol=1e-6))
print("std before rescale    :", out.std(dim=(1, 2, 3)).mean().item())
print("std after  rescale    :", classifier_free_guidance(eps_u, eps_c, 7.5, rescale=1.0).std(dim=(1, 2, 3)).mean().item())
print("std of conditional    :", eps_c.std(dim=(1, 2, 3)).mean().item())

In [ ]:
from torch_judge import check
check("cfg")